# Surrogate Model Training and Evaluation for the TIFON database

This template presents a complete demonstrator for the training and evaluation of surrogate models applied to the SHARP-Raptor test case.  

The problem consists of predicting the pressure coefficient field on the airfoil surface based on a set of input parameters.  

All the models compatible with this template are implemented within the **pyLOM** repository, which provides a unified framework for developing and evaluating machine learning models for aeronautical applications.

## Notebook Outline

0. Setting general configurations

1. Loading the datasets  
    1.1. Dataset loading and preprocessing utilities  
    1.2. Dataset construction, splitting, and scaling
2. Surrogate model definition and training  
    2.1. Hyperparameter optimization  
    2.2. Model training  
3. Evaluation of the surrogate model  
    3.1. Evaluation utilities  
    3.2. Model evaluation and results export

## 0. Setting general configurations

Global variables and directory paths required by the workflow are initialized.

The necessary Python libraries are imported, the main dataset and case directories are defined, and the output folder structure is automatically created. Dedicated folders are generated for trained models, hyperparameter optimization results, and plots.

In [ ]:
import os, json, copy, optuna, torch, kaggle, zipfile, tempfile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pathlib import Path

import pyLOM

from pyLOM.NN import MLP

DATA_DIR = "./../data"
CASE_DIR = "./results/mlp_vanilla"

print("Current working directory:", Path.cwd())
pyLOM.NN.create_results_folder(CASE_DIR)
pyLOM.NN.create_results_folder(os.path.join(CASE_DIR,'models'))
pyLOM.NN.create_results_folder(os.path.join(CASE_DIR,'hyperparameters'))
pyLOM.NN.create_results_folder(os.path.join(CASE_DIR,'plots'))
pyLOM.NN.create_results_folder(os.path.join(CASE_DIR,'validation'))

In [ ]:
CASE_NAME       = "mlp_vanilla_baseline"
STUDY_NAME      = "optuna_study"
N_TRIALS        = 200
N_EPOCHS_MAX    = 300
SEED            = 42
FONTSIZE        = 14
N_WORKERS       = 5
MODEL_CLASS     = MLP
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Loading the datasets

The aerodynamic data required for training the surrogate models is loaded. The dataset consists of pressure coefficients and associated flight parameters, including angle of attack and Mach number.

### 1.1. Dataset loading and preprocessing utilities

In this step, two utility functions are defined to handle the download, loading, and preprocessing of the aerodynamic dataset from the Kaggle competition.

The `get_data` function downloads the competition dataset directly from Kaggle into a temporary directory, extracts it, and loads the training, test, airfoil geometry, and validation-split files into pandas DataFrames. The temporary directory is automatically removed once the data has been loaded into memory, so no dataset files are kept on disk.

The `load_as_pyLOM_dataset` function then converts a given subset of cases (selected via a list of `case_id` values) into a `pyLOM.NN.Dataset` object. Its input features are composed of the airfoil surface coordinates (`x`, `y`), while the output corresponds to the pressure coefficient (`cp`) field. The associated angle of attack (`aoa`) and Mach number (`mach`) are included as conditioning parameters for the learning process. Optional input/output scalers can be passed to normalize the data consistently across train, validation, and test splits.

In [ ]:
def get_data():

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_path = Path(tmp_dir)
        
        kaggle.api.competition_download_files(
            'sharp-raptor-challenge',
            path=tmp_path
        )
        
        zip_file = next(tmp_path.glob('*.zip'))
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(tmp_path)
        
        df_train = pd.read_csv(tmp_path / 'train.csv')
        df_test = pd.read_csv(tmp_path / 'test.csv')
        airfoil_df = pd.read_csv(tmp_path / 'airfoil_coordinates.csv')
        validation_ids = pd.read_csv(tmp_path / 'validation_cases.csv')

    return df_train, df_test, airfoil_df, validation_ids

def load_as_pyLOM_dataset(df, geom_df, ids=[], inputs_scaler=None, outputs_scaler=None):
    
    return pyLOM.NN.Dataset(
        variables_out  = (df.loc[df['case_id'].isin(ids), 'cp'].to_numpy() if 'cp' in df.columns else np.zeros((len(df))),),
        variables_in   = geom_df[['x','y']].to_numpy(),
        parameters     = [df.loc[df['case_id'].isin(ids), 'aoa'].unique(), df.loc[df['case_id'].isin(ids), 'mach'].unique()], 
        inputs_scaler  = inputs_scaler,
        outputs_scaler = outputs_scaler,
    )

### 1.2. Dataset construction, splitting, and scaling

In this step, the aerodynamic dataset is downloaded from Kaggle and loaded using the previously defined utility functions. The training cases are split into training and validation subsets based on a predefined list of validation `case_id`s, while the test cases are loaded separately from the competition's test set.

Input and output variables are normalized using Min-Max scaling. The same scalers are fitted on the training data and consistently applied across all dataset splits (training, validation, and test) to ensure numerical stability during training and coherence in the data representation.

A combined training set (`td_train_complete`), merging the training and validation subsets, is also built for later use (e.g. final model retraining once hyperparameters have been selected).

Finally, basic statistics are printed for each dataset split to verify their structure and distribution.

In [ ]:
df_train, df_test, airfoil_df, validation_ids = get_data()

inputs_scaler = pyLOM.NN.MinMaxScaler()
outputs_scaler = pyLOM.NN.MinMaxScaler()

train_ids = np.setdiff1d(df_train['case_id'].unique(), validation_ids['case_id'].unique())
td_train = load_as_pyLOM_dataset(df_train, airfoil_df, ids=train_ids, inputs_scaler=inputs_scaler, outputs_scaler=outputs_scaler)

valid_ids = validation_ids['case_id']
td_val = load_as_pyLOM_dataset(df_train, airfoil_df, ids=valid_ids, inputs_scaler=inputs_scaler, outputs_scaler=outputs_scaler)

td_train_complete = copy.copy(td_train) + copy.copy(td_val)

train_ids = df_test['case_id'].unique()
td_test = load_as_pyLOM_dataset(df_test, airfoil_df, ids=train_ids, inputs_scaler=inputs_scaler, outputs_scaler=outputs_scaler)

# Print dataset stats
td_train.print_stats(dataset_name='Train')
td_train_complete.print_stats(dataset_name='Train Complete')
td_val.print_stats(dataset_name='Validation')
td_test.print_stats(dataset_name='Test')

## 2. Surrogate model definition and training

In this section, a the surrogate model from the `pyLOM.NN` module is trained to predict the pressure coefficient distribution over the airfoil surface from the input features. The objective is to learn a mapping between the aerodynamic conditions and the corresponding pressure response.

### 2.1. Hyperparameter optimization

In this step, the model hyperparameters are optimized according to the settings defined in the configuration file.

The optimization process is performed using Optuna through the pyLOM optimization pipeline. If no previous optimization results are found, a new hyperparameter search is executed and the corresponding studies and optimal parameter configurations are stored in the `hyperparameters` directory.

Once the optimization stage is completed, the best hyperparameters found during the study are automatically loaded and incorporated into the final training configuration used for model training.

In [ ]:
model_name = CASE_NAME
study_name = STUDY_NAME

hyperparameters_file = os.path.join(CASE_DIR, "hyperparameters", f"{study_name}.json")
optuna_file = os.path.join(CASE_DIR, "hyperparameters", "optuna_studies.db")
optuna_manager = pyLOM.NN.OptunaStudyManager(storage="sqlite:///" + optuna_file)

if not os.path.exists(hyperparameters_file):

    optimizer = pyLOM.NN.OptunaOptimizer(
        optimization_params={
        "optimizer_class": torch.optim.AdamW,
        "loss_fn": torch.nn.MSELoss(),
        "activation": torch.nn.functional.elu,
        "initialization": torch.nn.init.kaiming_normal_,
        "initialization_kwargs": {"nonlinearity": "relu"},
        "scheduler_class": torch.optim.lr_scheduler.StepLR,
        "scheduler_kwargs": {"step_size": 1, "gamma": (0.9, 0.999)},
        "dataloader_kwargs": {"num_workers": N_WORKERS, "prefetch_factor": 5, "persistent_workers": True},
        "lr": (1e-4, 1e-2),
        "batch_size": (64, 2048),
        "hidden_size": (64, 256),
        "n_layers": (2, 10),
        "p_dropout": (1e-5, 0.1),
        "print_rate_epoch": 0,
        "epochs": (2, N_EPOCHS_MAX),
        "device": DEVICE,
        "seed": SEED,
        "model_name": CASE_NAME,
        "save_logs_path": os.path.join(CASE_DIR, 'models'),
    },
        n_trials=optuna_manager.remaining_trials(study_name, N_TRIALS),
        direction="minimize",
        pruner=optuna.pruners.HyperbandPruner(min_resource=1, max_resource=N_EPOCHS_MAX, reduction_factor=3),
        sampler = optuna.samplers.QMCSampler(seed = SEED, warn_independent_sampling=False),
        save_dir=str(CASE_DIR + '/hyperparameters'),
        storage="sqlite:///" + optuna_file,
        study_name=study_name,
    )

    pipeline = pyLOM.NN.Pipeline(
        train_dataset=td_train,
        test_dataset=td_test,
        valid_dataset=td_val,
        model_class=MODEL_CLASS,
        optimizer=optimizer,
    )

    results = pipeline.run()
    model = pipeline.model

print("\nLoading hyperparameters...")
training_params = pyLOM.NN.OptunaOptimizer.load_best_params(
    base_params={
        "optimizer_class": torch.optim.AdamW,
        "loss_fn": torch.nn.MSELoss(),
        "activation": torch.nn.functional.elu,
        "initialization": torch.nn.init.kaiming_normal_,
        "initialization_kwargs": {"nonlinearity": "relu"},
        "scheduler_class": torch.optim.lr_scheduler.StepLR,
        "scheduler_kwargs": {"step_size": 1},
        "dataloader_kwargs": {"num_workers": N_WORKERS, "prefetch_factor": 5, "persistent_workers": True},
        "print_rate_epoch": 1,
        "device": DEVICE,
        "seed": SEED,
        "model_name": CASE_NAME,
        "save_logs_path": os.path.join(CASE_DIR, 'models'),
    },
    json_path=hyperparameters_file,
    verbose=True,
)

### 2.2. Model training

The MLP model is trained using the training pipeline provided by the pyLOM framework. Before starting the training process, the code checks whether a trained model already exists on disk.

If a saved model is found, it is loaded together with its associated training results, allowing the workflow to resume without retraining. Otherwise, a new model is initialized using the specified architecture and training parameters, and the pipeline is executed. Once the training process is completed, the model weights are saved to disk for future reuse.

In [ ]:
N_EPOCHS = training_params['epochs']
model_file = os.path.join(CASE_DIR, 'models/' + model_name + '_' + str(N_EPOCHS).zfill(5) + '.pth')
results_file = os.path.join(CASE_DIR, 'models/training_results_' + model_name + '_' + str(N_EPOCHS).zfill(5) + '.npy')

if os.path.exists(model_file):
    print(f"\nLoading model from {model_file}")
    model = MODEL_CLASS.load(model_file, DEVICE)
    results = np.load(results_file, allow_pickle=True).item()
    pipeline = pyLOM.NN.Pipeline(
        train_dataset=td_train,
        test_dataset=td_test,
        valid_dataset=td_val,
        model=model,
        training_params=training_params,
    )

else:
    print("\nTraining model...")
    model = MODEL_CLASS(input_size=td_train[:][0].shape[1], output_size=td_train[:][1].shape[1], **training_params)
    
    pipeline = pyLOM.NN.Pipeline(
        train_dataset=td_train,
        test_dataset=td_test,
        valid_dataset=td_val,
        model=model,
        training_params=training_params,
    )
    results = pipeline.run()
    pipeline.model.save(model_file)
    model = pipeline.model

## 3. Evaluation of the surrogate model

In this section, the trained surrogate model is evaluated to assess its predictive performance and generalization capability. The evaluation is carried out on the training, validation, and test datasets using quantitative metrics and visualization tools.

The objective is to analyze the accuracy and consistency of the model when predicting the pressure coefficient distribution.

### 3.1. Evaluation utilities

A set of auxiliary functions is defined to support the evaluation process. These utilities include:

- Visualization of training and validation loss curves to analyze convergence behavior.
- Scatter plots comparing true versus predicted values to assess regression quality.

In [ ]:
def plot_train_test_loss(train_loss, test_loss, path):
    """
    Auxiliary function to plot the training and test loss
    """
    total_epochs = len(train_loss)
    total_iters = len(train_loss)
    N = total_iters // total_epochs
    epoch = np.arange(1, total_epochs + 1)
    train_loss = np.mean(np.array(train_loss).reshape(-1,N), axis=1)
    train_loss = np.array(train_loss)
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(epoch, train_loss, label=f'Train Loss', linestyle='-', color='blue')
    ax.plot(epoch, test_loss, label=f'Test Loss', linestyle='--', color='red')

    ax.set_xlabel("Epoch", fontsize=FONTSIZE)
    ax.set_ylabel("Loss", fontsize=FONTSIZE)

    ax.set_yscale('log')
    ax.legend(frameon=True, loc='upper right', fontsize=FONTSIZE, borderpad=0.5, fancybox=True)
    ax.minorticks_on()
    ax.tick_params(axis='both', which='major', length=7, width=1, direction='in')
    ax.tick_params(axis='both', which='minor', length=4, width=0.8, direction='in')

    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.show()


def plot_true_vs_pred(y_true, y_pred, path):
    """
    Auxiliary function to plot true vs predicted values
    """
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_true, y_pred, alpha=0.5)
    ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)

    ax.set_xlabel("True values", fontsize=FONTSIZE)
    ax.set_ylabel("Predicted values", fontsize=FONTSIZE)

    ax.minorticks_on()
    ax.tick_params(axis='both', which='major', length=7, width=1, direction='in')
    ax.tick_params(axis='both', which='minor', length=4, width=0.8, direction='in')

    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.show()

### 3.2. Model evaluation and results export

The trained model is evaluated on the training, validation, and test datasets using the pyLOM evaluation pipeline. Performance metrics are computed, and predictions are compared against ground truth values.

Loss curves are analyzed to assess convergence, and a diagnostic plot is generated for the test dataset.

In [ ]:
# Number of model trainable parameters
print(f'Number of model trainable parameters:', model._count_n_parameters())

# Train and validation loss curves
plot_train_test_loss(results['train_loss'], results['test_loss'], os.path.join(CASE_DIR, "plots", "train_test_loss.png"))
validation_logs = {"train_loss": results["train_loss"], "test_loss": results['test_loss']}

# Evaluation of the model
evaluator = pyLOM.NN.RegressionEvaluator()
evaluation_params = {
    "batch_size": 2**15,
    "dataloader_kwargs": {
        "num_workers": N_WORKERS,
        "prefetch_factor": 5,
        "persistent_workers": True
    }
}

print(f'-'*50)
print(f'Train dataset evaluation:')
_, variables_train = pipeline.evaluate(set_to_use="train", evaluators=[evaluator], scalers=[inputs_scaler, outputs_scaler], evaluation_params=evaluation_params)

print(f'-'*50)
print(f'Validation dataset evaluation:')
_, variables_valid = pipeline.evaluate(set_to_use="valid", evaluators=[evaluator], scalers=[inputs_scaler, outputs_scaler], evaluation_params=evaluation_params)

print(f'-'*50)
print(f'Test dataset evaluation:')
_, variables_test = pipeline.evaluate(set_to_use="test", evaluators=[evaluator], scalers=[inputs_scaler, outputs_scaler], evaluation_params=evaluation_params)
print(f'-'*50)

# True vs predicted plot for the test dataset
plot_true_vs_pred(variables_valid[1], variables_valid[2], os.path.join(CASE_DIR, "plots", "true_vs_pred_test.png"))

In [ ]:
df_test["cp"] = variables_test[2]
submission = df_test[["id", "cp"]]
submission.to_csv(os.path.join(CASE_DIR, "validation", "mlp_vanilla_baseline.csv"), index=False)

submission.head()